In [1]:
import numpy as np    
import pandas as pd


In [8]:
df=pd.read_csv('land_data_after_feature_engineering_and_encoding.csv')
df.sample()

,district,road_type,land_size_aana,price_per_aana,is_large_plot,road_width_feet,is_wide_road,facing,source,log_land,neighborhood_encoded
230,2,1,5.0,3050000.0,0,13.0,0,7,0,1.791759,14.798337


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [5]:
# ─────────────────────────────────────────
# STEP 1 — DEFINE X AND Y
# ─────────────────────────────────────────
X = df.drop(columns=['price_per_aana'])
y = np.log1p(df['price_per_aana'])

print(f"Features: {X.shape[1]}")
print(f"Samples:  {X.shape[0]}")
print(f"Target mean: {y.mean():.4f}")
print(f"Target min:  {y.min():.4f}")
print(f"Target max:  {y.max():.4f}")

Features: 10
Samples:  4063
Target mean: 15.2932
Target min:  13.1224
Target max:  16.8112


In [9]:
# ─────────────────────────────────────────
# STEP 2 — TRAIN TEST SPLIT
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


Training samples: 3250
Testing samples:  813


In [11]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

In [12]:
# ─────────────────────────────────────────
# STEP 3 — TRAIN ALL MODELS
# ─────────────────────────────────────────
models = {
    'Linear Regression':   LinearRegression(),
    'Ridge':               Ridge(alpha=1.0),
    'Lasso':               Lasso(alpha=0.001),
    'Decision Tree':       DecisionTreeRegressor(random_state=42),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':             XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM':            lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost':            CatBoostRegressor(n_estimators=100, random_state=42, verbose=0),
    'KNN':                 KNeighborsRegressor(n_neighbors=5),
    'SVR':                 SVR(kernel='rbf')
}

In [15]:
# Scale for linear models
from sklearn.preprocessing import StandardScaler
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)

results = []
linear_models = ['Linear Regression', 'Ridge', 'Lasso', 'KNN', 'SVR']

for name, model in models.items():
    if name in linear_models:
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results.append({
        'Model':      name,
        'RMSE (log)': round(rmse, 4),
        'MAE (log)':  round(mae, 4),
        'R² Score':   round(r2, 4),
        'Error %':    f"{(np.exp(mae) - 1) * 100:.1f}%"
    })
    print(f"✅ {name} done")

✅ Linear Regression done
✅ Ridge done
✅ Lasso done
✅ Decision Tree done
✅ Random Forest done
✅ Gradient Boosting done
✅ XGBoost done
✅ LightGBM done
✅ CatBoost done
✅ KNN done
✅ SVR done


In [16]:
# ─────────────────────────────────────────
# STEP 4 — COMPARE
# ─────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False)
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))


=== Model Comparison ===
            Model  RMSE (log)  MAE (log)  R² Score Error %
            Lasso      0.3744     0.2425    0.6091   27.4%
Linear Regression      0.3748     0.2426    0.6083   27.5%
            Ridge      0.3748     0.2426    0.6083   27.5%
              SVR      0.3812     0.2466    0.5946   28.0%
Gradient Boosting      0.3820     0.2466    0.5929   28.0%
         CatBoost      0.3869     0.2497    0.5824   28.4%
         LightGBM      0.3902     0.2516    0.5752   28.6%
          XGBoost      0.4119     0.2691    0.5268   30.9%
    Random Forest      0.4127     0.2713    0.5250   31.2%
              KNN      0.4143     0.2807    0.5213   32.4%
    Decision Tree      0.5001     0.3345    0.3024   39.7%


In [17]:
from sklearn.model_selection import RandomizedSearchCV


In [18]:
param_grids = {
    'Lasso': {
        'model': Lasso(),
        'params': {
            'alpha': [1e-5, 1e-4, 1e-3, 0.01, 0.1, 1.0]
        }
    },
    'Ridge': {
        'model': Ridge(),
        'params': {
            'alpha': [1.0, 5.0, 10.0, 20.0, 50.0, 100.0]
        }
    },
    'SVR': {
        'model': SVR(),
        'params': {
            'C':       [1, 10, 100, 1000],
            'epsilon': [0.001, 0.01, 0.1],
            'kernel':  ['linear', 'rbf']
        }
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, verbosity=0),
        'params': {
            'n_estimators':     [500, 1000],
            'learning_rate':    [0.01, 0.03, 0.05],
            'max_depth':        [4, 6, 8],
            'subsample':        [0.8, 0.9],
            'colsample_bytree': [0.6, 0.8],
            'gamma':            [0, 0.1, 0.2]
        }
    },
    'CatBoost': {
        'model': CatBoostRegressor(random_state=42, verbose=0),
        'params': {
            'iterations':          [1000, 1500],
            'learning_rate':       [0.01, 0.05, 0.1],
            'depth':               [6, 8, 10],
            'l2_leaf_reg':         [3, 5, 9],
            'bagging_temperature': [0, 1]
        }
    }
}


In [21]:
tuned_models  = {}
tuned_results = []

for name, config in param_grids.items():
    print(f"🔍 Tuning {name}...")

    # Set n_iter based on model type
    n_iter = 20 if name in ['Lasso', 'Ridge', 'SVR'] else 50

    search = RandomizedSearchCV(
        estimator=config['model'],
        param_distributions=config['params'],
        n_iter=n_iter,
        cv=5,
        scoring='r2',
        random_state=42,
        n_jobs=-1
    )

    # Use scaled data for linear models and SVR
    if name in ['Lasso', 'Ridge', 'SVR']:
        search.fit(X_train_sc, y_train)
        tuned_models[name] = search.best_estimator_
        y_pred = tuned_models[name].predict(X_test_sc)
    else:
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_
        y_pred = tuned_models[name].predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    tuned_results.append({
        'Model':       name,
        'RMSE (log)':  round(rmse, 4),
        'MAE (log)':   round(mae, 4),
        'R² Score':    round(r2, 4),
        'Error %':     f"{(np.exp(mae) - 1) * 100:.1f}%",
        'Best Params': search.best_params_
    })

    print(f"✅ {name} done — R²: {r2:.4f}")
print('HyperParameter Tuning Done. ✅')

🔍 Tuning Lasso...


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 6 is smaller than n_iter=20. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


✅ Lasso done — R²: 0.6084
🔍 Tuning Ridge...
✅ Ridge done — R²: 0.6083
🔍 Tuning SVR...


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 6 is smaller than n_iter=20. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


✅ SVR done — R²: 0.6072
🔍 Tuning Gradient Boosting...
✅ Gradient Boosting done — R²: 0.6001
🔍 Tuning XGBoost...
✅ XGBoost done — R²: 0.6002
🔍 Tuning CatBoost...
✅ CatBoost done — R²: 0.6091
HyperParameter Tuning Done. ✅


In [22]:
# ─────────────────────────────────────────
# COMPARE RESULTS
# ─────────────────────────────────────────
tuned_df = pd.DataFrame(tuned_results).sort_values('R² Score', ascending=False)

print("\n=== Tuned Model Comparison (Round 2) ===")
print(tuned_df[['Model', 'RMSE (log)', 'MAE (log)', 'R² Score', 'Error %']].to_string(index=False))

print("\n=== Best Parameters ===")
for result in tuned_results:
    print(f"\n{result['Model']}:")
    for param, value in result['Best Params'].items():
        print(f"  {param}: {value}")


=== Tuned Model Comparison (Round 2) ===
            Model  RMSE (log)  MAE (log)  R² Score Error %
         CatBoost      0.3743     0.2420    0.6091   27.4%
            Lasso      0.3747     0.2426    0.6084   27.5%
            Ridge      0.3748     0.2426    0.6083   27.5%
              SVR      0.3753     0.2409    0.6072   27.2%
          XGBoost      0.3786     0.2428    0.6002   27.5%
Gradient Boosting      0.3786     0.2439    0.6001   27.6%

=== Best Parameters ===

Lasso:
  alpha: 0.0001

Ridge:
  alpha: 10.0

SVR:
  kernel: linear
  epsilon: 0.01
  C: 1

Gradient Boosting:
  subsample: 0.7
  n_estimators: 100
  min_samples_split: 5
  max_depth: 3
  learning_rate: 0.05

XGBoost:
  subsample: 0.8
  n_estimators: 500
  max_depth: 3
  learning_rate: 0.01
  colsample_bytree: 0.7

CatBoost:
  learning_rate: 0.05
  l2_leaf_reg: 5
  iterations: 200
  depth: 4


In [25]:
# ─────────────────────────────────────────
# FEATURE ENGINEERING — LAND DATASET
# ─────────────────────────────────────────
df_ml_land_v2 = df.copy()

# 1 — Road quality score combining road_type and road_width_feet
# road_type is already encoded — combine with width for a unified score
df_ml_land_v2['road_quality_score'] = (
    df_ml_land_v2['road_type'] * 0.6 +
    np.log1p(df_ml_land_v2['road_width_feet']) * 0.4
)

# 2 — Neighborhood × district interaction
# Captures micro-market within each district
df_ml_land_v2['neighborhood_x_district'] = (
    df_ml_land_v2['neighborhood_encoded'] * df_ml_land_v2['district']
)

# 3 — Land size category
# Small/Medium/Large plot flag — captures non-linear size effect
df_ml_land_v2['plot_size_category'] = pd.cut(
    df_ml_land_v2['land_size_aana'],
    bins=[0, 4, 10, 20, 50],
    labels=[1, 2, 3, 4]
).astype(int)

# 4 — Price zone proxy from neighborhood encoding
# Bin neighborhood_encoded into tiers
df_ml_land_v2['location_tier'] = pd.qcut(
    df_ml_land_v2['neighborhood_encoded'],
    q=5, labels=[1, 2, 3, 4, 5]
).astype(int)

# 5 — Large plot penalty
# Large plots are cheaper per aana — interaction with neighborhood
df_ml_land_v2['large_plot_x_neighborhood'] = (
    df_ml_land_v2['is_large_plot'] * df_ml_land_v2['neighborhood_encoded']
)

# ─────────────────────────────────────────
# VERIFY
# ─────────────────────────────────────────
print(f"Features before: {df.shape[1]}")
print(f"Features after:  {df_ml_land_v2.shape[1]}")
print(f"New features: {list(df_ml_land_v2.columns)}")
print(f"\nNulls: {df_ml_land_v2.isnull().sum().sum()}")

Features before: 11
Features after:  16
New features: ['district', 'road_type', 'land_size_aana', 'price_per_aana', 'is_large_plot', 'road_width_feet', 'is_wide_road', 'facing', 'source', 'log_land', 'neighborhood_encoded', 'road_quality_score', 'neighborhood_x_district', 'plot_size_category', 'location_tier', 'large_plot_x_neighborhood']

Nulls: 0


In [26]:
# ─────────────────────────────────────────
# RETRAIN WITH NEW FEATURES
# ─────────────────────────────────────────
X_v2 = df_ml_land_v2.drop(columns=['price_per_aana'])
y_v2 = np.log1p(df_ml_land_v2['price_per_aana'])

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42
)

In [27]:
# Scale for linear models
X_train_v2_sc = scaler.fit_transform(X_train_v2)
X_test_v2_sc  = scaler.transform(X_test_v2)

In [28]:
models_v2 = {
    'Lasso':    Lasso(alpha=0.0001),
    'Ridge':    Ridge(alpha=10.0),
    'SVR':      SVR(kernel='linear', C=1, epsilon=0.01),
    'XGBoost':  XGBRegressor(n_estimators=500, learning_rate=0.01,
                             max_depth=3, subsample=0.8,
                             colsample_bytree=0.7, random_state=42, verbosity=0),
    'CatBoost': CatBoostRegressor(iterations=200, learning_rate=0.05,
                                  depth=4, l2_leaf_reg=5,
                                  random_state=42, verbose=0)
}

results_v2 = []

for name, model in models_v2.items():
    if name in ['Lasso', 'Ridge', 'SVR']:
        model.fit(X_train_v2_sc, y_train_v2)
        y_pred = model.predict(X_test_v2_sc)
    else:
        model.fit(X_train_v2, y_train_v2)
        y_pred = model.predict(X_test_v2)

    rmse = np.sqrt(mean_squared_error(y_test_v2, y_pred))
    mae  = mean_absolute_error(y_test_v2, y_pred)
    r2   = r2_score(y_test_v2, y_pred)

    results_v2.append({
        'Model':      name,
        'RMSE (log)': round(rmse, 4),
        'MAE (log)':  round(mae, 4),
        'R² Score':   round(r2, 4),
        'Error %':    f"{(np.exp(mae) - 1) * 100:.1f}%"
    })
    print(f"✅ {name} done — R²: {r2:.4f}")


✅ Lasso done — R²: 0.6083
✅ Ridge done — R²: 0.6083
✅ SVR done — R²: 0.6068
✅ XGBoost done — R²: 0.6005
✅ CatBoost done — R²: 0.6117


In [29]:
# ─────────────────────────────────────────
# COMPARE BEFORE VS AFTER
# ─────────────────────────────────────────
results_v2_df = pd.DataFrame(results_v2).sort_values('R² Score', ascending=False)

print("\n=== Before Feature Engineering ===")
print(tuned_df[['Model', 'R² Score', 'Error %']].to_string(index=False))

print("\n=== After Feature Engineering ===")
print(results_v2_df[['Model', 'R² Score', 'Error %']].to_string(index=False))


=== Before Feature Engineering ===
            Model  R² Score Error %
         CatBoost    0.6091   27.4%
            Lasso    0.6084   27.5%
            Ridge    0.6083   27.5%
              SVR    0.6072   27.2%
          XGBoost    0.6002   27.5%
Gradient Boosting    0.6001   27.6%

=== After Feature Engineering ===
   Model  R² Score Error %
CatBoost    0.6117   27.4%
   Lasso    0.6083   27.5%
   Ridge    0.6083   27.5%
     SVR    0.6068   27.3%
 XGBoost    0.6005   27.6%


In [30]:
# Save best model
import joblib
joblib.dump(models_v2['CatBoost'], 'catboost_land_model_final.pkl')

# Save feature engineered dataset
df_ml_land_v2.to_csv('land_features_final_modeled.csv', index=False)

print("✅ CatBoost land model saved")
print("✅ Feature engineered dataset saved")

# Verify
loaded = joblib.load('catboost_land_model_final.pkl')
preds  = loaded.predict(X_test_v2[:3])
print(f"✅ Sample predictions (price per aana): {np.expm1(preds).astype(int)}")

✅ CatBoost land model saved
✅ Feature engineered dataset saved
✅ Sample predictions (price per aana): [2033292 4365185 2376561]
